<img align="left" src = https://noirlab.edu/public/media/archives/logos/svg/logo250.svg width=250 style="background-color:white; padding:10px" alt="Rubin Observatory logo, a graphical representation of turning stars into data."><br><b>Introduction to Jupyter Notebooks for Data Preview 0.2</b> <br>
Contact author: Gloria Fonseca Alvarez <br>
Last verified to run: 2025-06-10 <br>
LSST Science Pipelines version: Weekly 2025_17 <br>
Container size: medium <br>
Targeted learning level: beginner <br>

**Description:** An introduction to using Jupyter Notebooks and Rubin Python packages to access LSST data products (images and catalogs).

**Skills:** Use the TAP service to retrieve Object catalog data. Use the Butler to retrieve and display a deepCoadd image.

**LSST Data Products:** TAP dp02_dc2_catalogs.Object table. Butler deepCoadd image.

**Packages:** lsst.rsp.get_tap_service, lsst.rsp.retrieve_query, lsst.daf.butler, lsst.afw.display, lsst.geom, pandas, matplotlib

**Credit:** This demo is based on notebook 01_Introduction_to_DP02 developed by Melissa Graham and other notebooks by the Rubin Community Science Team.

**Get Support:**
Find DP0-related documentation and resources at <a href="https://dp0.lsst.io">dp0.lsst.io</a>. Questions are welcome as new topics in the <a href="https://community.lsst.org/c/support/dp0">Support - Data Preview 0 Category</a> of the Rubin Community Forum. Rubin staff will respond to all questions posted there.

## 1.0. Introduction¶

This Jupyter Notebook provides an introduction to how to access catalog and image data products.

This Notebook uses the Data Preview 0.2 (DP0.2) data set. This data set uses a subset of the DESC's Data Challenge 2 (DC2) simulated images, which have been *reprocessed* by Rubin Observatory using Version 23 of the LSST Science Pipelines. More information about the simulated data can be found in the <a href="https://ui.adsabs.harvard.edu/abs/2021ApJS..253...31L/abstract">DESC's DC2 paper</a> and in the <a href="https://dp0-2.lsst.io">DP0.2 data release documentation</a>.

### 1.1. Package Imports

Import commonly used python packages and packages from the  <a href="https://pipelines.lsst.io/">LSST Science Pipelines</a>.

In [ ]:
import numpy
import matplotlib
import matplotlib.pyplot as plt
import pandas
from lsst.rsp import get_tap_service, retrieve_query
import lsst.daf.butler as dafButler
import lsst.geom
import lsst.afw.display as afwDisplay

#### 1.2. Instantiate the TAP service

Start the TAP service to be used for catalog queries.

In [ ]:
service = get_tap_service("tap")

#### 1.3. Instantiate the Butler

In [ ]:
butler = dafButler.Butler('dp02', collections='2.2i/runs/DP0.2')

## 2.0. Catalog Data

Define the central coordinates of the simulation 

In [ ]:
use_center_coords = "62, -37"

#### 2.1 Retrive data for 100 objects

Create a query to get coordinates and properties of 100 objects from the Object catalog. Use a 0.01 degree search radius. For a more detailed introduction on how to write TAP queries, see notebook 02b_Catalog_Queries_with_TAP.

In [ ]:
query = "SELECT TOP 100 "\
        "coord_ra, coord_dec, r_calibFlux, r_cModelFlux, "\
        "r_extendedness, detect_isPrimary "\
        "FROM dp02_dc2_catalogs.Object "\
        "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), "\
        "CIRCLE('ICRS', " + use_center_coords + ", 0.01)) = 1 "

Run the query and save the results as a table.

In [ ]:
results = service.search(query)
results_table = results.to_table()

In [ ]:
results_table

#### 2.2 Retrive data for 10,0000 point-like objects

Create a query to get coordinates and properties of 10,0000 objects from the Object catalog. Use a 1 degree search radius. Convert fluxes to magnitudes with the function ``scisql_nanojanskyToAbMag()``. Constrain the search to objects that are deblended (detect_isPrimary = 1), bright (calibFlux > 360), and point-like (extendedness = 0)

In [ ]:
query = "SELECT TOP 10000 "\
        "coord_ra, coord_dec, "\
        "scisql_nanojanskyToAbMag(g_calibFlux) as g_calibMag, "\
        "scisql_nanojanskyToAbMag(r_calibFlux) as r_calibMag, "\
        "scisql_nanojanskyToAbMag(i_calibFlux) as i_calibMag "\
        "FROM dp02_dc2_catalogs.Object "\
        "WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec), "\
        "CIRCLE('ICRS', "+use_center_coords+", 1.0)) = 1 "\
        "AND detect_isPrimary = 1 "\
        "AND g_calibFlux > 360 "\
        "AND r_calibFlux > 360 "\
        "AND i_calibFlux > 360 "\
        "AND g_extendedness = 0 "\
        "AND r_extendedness = 0 "\
        "AND i_extendedness = 0"

In [ ]:
results = service.search(query)
results_table = results.to_table()

In [ ]:
results_table

### 2.3 Make color-magnitude and color-color diagrams

Save the table to a pandas dataframe for easy access to contents.

In [ ]:
data = results_table.to_pandas()

Create the plots.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(data['g_calibMag'].values - data['r_calibMag'].values,
         data['r_calibMag'].values, 'o', ms=2, alpha=0.2)

ax[0].set_xlabel('(g - r)', fontsize=16)
ax[0].set_ylabel('r mag', fontsize=16)
ax[0].invert_yaxis()
ax[0].minorticks_on()

ax[1].plot(data['g_calibMag'].values - data['r_calibMag'].values,
         data['r_calibMag'].values - data['i_calibMag'].values, 'o', ms=2, alpha=0.2)

ax[1].set_xlabel('(g - r)', fontsize=16)
ax[1].set_ylabel('(r - i)', fontsize=16)
ax[1].minorticks_on()

plt.show()

## 3.0. Image Data

#### 3.1 Retrieve a deepCoadd image

Define the coordinates for a known galaxy cluster in the DP0.2 dataset.

In [ ]:
cluster_ra = 55.745834
cluster_dec = -32.269167

Find the tract and patch for the cluster's coordinates. 

In [ ]:
spherePoint = lsst.geom.SpherePoint(cluster_ra*lsst.geom.degrees,
                                       cluster_dec*lsst.geom.degrees)
skymap = butler.get('skyMap')
tract = skymap.findTract(spherePoint)
patch = tract.findPatch(spherePoint)

cluster_tract = tract.tract_id
cluster_patch = patch.getSequentialIndex()

print("The cluster is in tract ", cluster_tract,", patch ", cluster_patch)

Define a dataId using the tract, patch, and desired band.

In [ ]:
dataId = {'band': 'i', 'tract': cluster_tract, 'patch': cluster_patch}

Retrieve the deepCoadd.

In [ ]:
deepCoadd = butler.get('deepCoadd', dataId=dataId)

#### 3.2. Display the image.

Image data retrieved with the butler can be displayed several different ways. A simple option is to use the LSST Science Pipelines package afwDisplay.

##### 3.2.1. Use afwDisplay to show the image data with matplotlib.

In [ ]:
afwDisplay.setDefaultBackend('matplotlib')
fig = plt.figure(figsize=(10, 8))
afw_display = afwDisplay.Display(1)
afw_display.scale('asinh', 'zscale')
afw_display.mtv(deepCoadd.image)
plt.gca().axis('on')
plt.show()

##### 3.2.2. Use afwDisplay to show the image data with Firefly.

Use the [Firefly](https://pipelines.lsst.io/v/daily/modules/lsst.display.firefly/index.html) interface to interact with the image. 

In [ ]:
afwDisplay.setDefaultBackend('firefly')
afw_display = afwDisplay.Display(frame=1)
afw_display.mtv(deepCoadd)
afw_display.setMaskTransparency(100)